In [ ]:
from langchain_core.documents import Document

documents = [

    Document(
        page_content="""
        LangGraph is a framework for building stateful applications
        with large language models. Applications are represented as
        graphs containing nodes and edges.
        """,
        metadata={"topic": "LangGraph"}
    ),

    Document(
        page_content="""
        LangGraph allows applications to maintain state between
        different nodes. Nodes can read the current state and
        update it during graph execution.
        """,
        metadata={"topic": "State"}
    ),

    Document(
        page_content="""
        LangGraph persistence allows applications to save graph state.
        Checkpointers store snapshots of the state during execution.
        This allows a graph to resume from a previous state.
        """,
        metadata={"topic": "Persistence"}
    ),

    Document(
        page_content="""
        LangGraph can maintain conversational memory by using
        persistence and checkpoints. Previous conversation state
        can be retrieved when the user returns to a conversation.
        """,
        metadata={"topic": "Memory"}
    ),

    Document(
        page_content="""
        LangGraph supports human-in-the-loop workflows. A graph can
        pause execution and wait for human approval before continuing.
        """,
        metadata={"topic": "Human-in-the-loop"}
    ),

    Document(
        page_content="""
        LangGraph can implement agentic RAG systems using nodes for
        retrieval, generation, tool execution and conditional routing.
        """,
        metadata={"topic": "RAG"}
    )
]

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOllama(model="llama3.2", temperature=0)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

multi_query_prompt = ChatPromptTemplate.from_template("""
You are an expert at generating search queries for a RAG system.

Given the user's question, generate 3 different search queries
that represent different aspects or interpretations of the question.

The goal is to improve document retrieval.

Original question:
{question}

Generate exactly 3 search queries.

Return each query on a separate line.
Do not number them.
Do not provide explanations.
""")

multi_query_chain = multi_query_prompt | llm

In [ ]:
question = "How does LangGraph handle memory?"

response = multi_query_chain.invoke({
    "question": question
})

queries = [
    q.strip()
    for q in response.content.split("\n")
    if q.strip()
]

print(queries)

In [ ]:
all_documents = []

for query in queries:

    docs = retriever.invoke(query)

    all_documents.extend(docs)

all_documents

In [ ]:
unique_documents = {}

for doc in all_documents:

    key = doc.page_content

    unique_documents[key] = doc

documents = list(unique_documents.values())
documents

In [ ]:
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful RAG assistant.

Answer the user's question using ONLY the provided context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say that there is not enough information.

Answer:
""")
answer_chain = answer_prompt | llm

context = "\n\n".join(
    doc.page_content
    for doc in documents
)

In [ ]:
answer = answer_chain.invoke({
    "context": context,
    "question": question
})

print(answer.content)

In [ ]:
# Multi-query using LangGraph documents and Ollama embeddings with Chroma vector store.

from typing import TypedDict, List
from langchain_core.documents import Document


class RAGState(TypedDict):

    question: str

    queries: List[str]

    documents: List[Document]

    answer: str


In [ ]:
def generate_queries(state: RAGState):

    response = multi_query_chain.invoke({
        "question": state["question"]
    })

    queries = [
        q.strip()
        for q in response.content.split("\n")
        if q.strip()
    ]

    return {
        "queries": queries
    }

def retrieve_documents(state: RAGState):

    all_documents = []

    for query in state["queries"]:

        docs = retriever.invoke(query)

        all_documents.extend(docs)

    return {
        "documents": all_documents
    }

def deduplicate_documents(state: RAGState):

    unique_documents = {}

    for doc in state["documents"]:

        key = doc.page_content

        unique_documents[key] = doc

    return {
        "documents": list(unique_documents.values())
    }

def generate_answer(state: RAGState):

    context = "\n\n".join(
        doc.page_content
        for doc in state["documents"]
    )

    response = answer_chain.invoke({
        "context": context,
        "question": state["question"]
    })

    return {
        "answer": response.content
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(RAGState)

builder.add_node(
    "generate_queries",
    generate_queries
)

builder.add_node(
    "retrieve_documents",
    retrieve_documents
)

builder.add_node(
    "deduplicate_documents",
    deduplicate_documents
)

builder.add_node(
    "generate_answer",
    generate_answer
)

builder.add_edge(
    START,
    "generate_queries"
)

builder.add_edge(
    "generate_queries",
    "retrieve_documents"
)

builder.add_edge(
    "retrieve_documents",
    "deduplicate_documents"
)

builder.add_edge(
    "deduplicate_documents",
    "generate_answer"
)

builder.add_edge(
    "generate_answer",
    END
)

graph = builder.compile()

graph

In [ ]:
result = graph.invoke({
    "question": "How does LangGraph handle memory?"
})
print("===== GENERATED QUERIES =====")

for query in result["queries"]:
    print("-", query)

print("\n===== RETRIEVED DOCUMENTS =====")

for doc in result["documents"]:

    print("\n---------------------")

    print(doc.page_content)

print("\n===== FINAL ANSWER =====")

print(result["answer"])